In [ ]:
import numpy as np
import jdaviz as jd

import astropy.units as u
from astropy.coordinates import ICRS
from astropy.modeling import models
from astropy.nddata import NDData
from astropy.wcs import WCS

from gwcs import WCS as GWCS
from gwcs import coordinate_frames as cf, wcs as gwcs_wcs

In [ ]:
n_images = 5
image_pixel_side = 5000
wcs_type = '{wcs_type_value}'  # noqa: F821
link_type = 'wcs'
batch_load = True
image_alpha = 1

In [ ]:
shape = (image_pixel_side, image_pixel_side)

# Roman plate scale:
pixel_scale = (0.11 * (u.arcsec / u.pixel)).to_value(u.deg / u.pixel)

def generate_wcs(rho=np.pi / 3):
    """
    rho = rotation wrt sky coordinate frame [radians]
    """
    # Example adapted from photutils:
    #   https://github.com/astropy/photutils/blob/
    #   2825356f1d876cacefb3a03d104a4c563065375f/photutils/datasets/make.py#L821
    
    shift_by_crpix = (models.Shift((-shape[1] / 2) + 1)
                      & models.Shift((-shape[0] / 2) + 1))
    
    cd_matrix = np.array([[-pixel_scale * np.cos(rho), pixel_scale * np.sin(rho)],
                          [pixel_scale * np.sin(rho), pixel_scale * np.cos(rho)]])
    
    rotation = models.AffineTransformation2D(cd_matrix, translation=[0, 0])
    rotation.inverse = models.AffineTransformation2D(
        np.linalg.inv(cd_matrix), translation=[0, 0])
    
    tan = models.Pix2Sky_TAN()
    celestial_rotation = models.RotateNative2Celestial(197.8925, -1.36555556, 180.0)
    
    det2sky = shift_by_crpix | rotation | tan | celestial_rotation
    det2sky.name = 'linear_transform'
    
    detector_frame = cf.Frame2D(name='detector', axes_names=('x', 'y'), unit=(u.pix, u.pix))
    
    sky_frame = cf.CelestialFrame(reference_frame=ICRS(), name='icrs', unit=(u.deg, u.deg))
    
    pipeline = [(detector_frame, det2sky), (sky_frame, None)]
    
    gwcs = gwcs_wcs.WCS(pipeline)
    gwcs.bounding_box = [(0, shape[0]), (0, shape[1])]
    
    if wcs_type == 'wcs':
        wcs = WCS(gwcs.to_fits()[0])
    else:
        wcs = gwcs

    return wcs

rng = np.random.default_rng(42)

images = [
    NDData(
        data=rng.uniform(0, 2**16, size=shape),
        wcs=generate_wcs(i * rng.uniform(0, np.pi/2))
    ) for i in range(n_images)
]

In [ ]:
if batch_load:
    with jd.batch_load():
        for i, image in enumerate(images):
            jd.load(image, format='Image', data_label=f"{i}")

else: 
    for i, image in enumerate(images):
        jd.load(image, format='Image', data_label=f"{i}")


jd.show()

In [ ]:
plot_opts = jd.plugins['Plot Options']
plot_opts.image_color_mode = 'Colormap'

for layer in jd.viewers['Image']._obj.glue_viewer.state.layers:
    layer.cmap_bad = (0, 0, 0, 0)
    layer.alpha = image_alpha

In [ ]:
def to_zoom_center(pixels):
    """
    Convert zoom center from pixels to degrees if WCS linked
    """
    if link_type == 'pixel':
        return pixels

    result = images[0].wcs.pixel_to_world_values(*pixels)
    return tuple(float(val) for val in result)

def to_zoom_radius(pixels):
    """
    Convert zoom radius from pixels to degrees if WCS linked
    """
    if link_type == 'pixel':
        return pixels

    return pixel_scale * pixels

In [ ]:
orientation = jd.plugins['Orientation']

if link_type == 'wcs':
    orientation.align_by = link_type.upper()

In [ ]:
# zoom in close
plot_opts.zoom_radius = to_zoom_radius(10)

In [ ]:
# zoom out
plot_opts.zoom_radius = to_zoom_radius(100)

In [ ]:
# zoom out
plot_opts.zoom_radius = to_zoom_radius(1_000)

In [ ]:
# zoom out
plot_opts.zoom_radius = to_zoom_radius(2_000)

In [ ]:
# zoom out
plot_opts.zoom_radius = to_zoom_radius(5_000)

In [ ]:
# zoom out
plot_opts.zoom_radius = to_zoom_radius(15_000)

In [ ]:
# pan and zoom to ~1/4 of the first image
plot_opts.layer = plot_opts.layer.choices[0]
x, y = to_zoom_center((image_pixel_side / 4, image_pixel_side / 4))
plot_opts.zoom_center_x = x
plot_opts.zoom_center_y = y
plot_opts.zoom_radius = to_zoom_radius(shape[0] / 4)

In [ ]:
# pan to an out-of-viewer portion of the image at the same zoom level
plot_opts.layer = plot_opts.layer.choices[0]
x, y = to_zoom_center((3 * image_pixel_side / 4, 3 * image_pixel_side / 4))
plot_opts.zoom_center_x = x
plot_opts.zoom_center_y = y
plot_opts.zoom_radius = to_zoom_radius(shape[0] / 4)